In [0]:
%pip install psxdata
dbutils.library.restartPython()

Looking in indexes: [REDACTED]
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 0.0/5.0 MB ? eta -:--:--   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.0/5.0 MB 83.7 MB/s eta 0:00:00
Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
import psxdata
from pyspark.sql import SparkSession

tickers = ["UBL", "MCB", "MEBL", "ENGRO", "LUCK"]

pdf_list = []
for t in tickers:
    df = psxdata.stocks(t, start="2024-01-01", end="2026-08-18")
    df["symbol"] = t
    pdf_list.append(df)

import pandas as pd
raw_pdf = pd.concat(pdf_list)

# Convert to Spark and write as a Delta (Bronze) table
bronze_df = spark.createDataFrame(raw_pdf)
bronze_df.write.format("delta").mode("overwrite").saveAsTable("psx_bronze")

OHLC constraint violated in 2 row(s) — flagged
Dates are not in chronological order
OHLC constraint violated in 4 row(s) — flagged
Dates are not in chronological order
OHLC constraint violated in 3 row(s) — flagged
Dates are not in chronological order
OHLC constraint violated in 3 row(s) — flagged
Dates are not in chronological order
Dates are not in chronological order


In [0]:
%sql
SELECT * FROM psx_bronze LIMIT 10

date,open,high,low,close,volume,is_anomaly,symbol
2025-06-10T00:00:00.000Z,282.74,284.33,280.5,281.57,286846,false,MCB
2025-06-05T00:00:00.000Z,284.07,284.07,282.03,282.74,268278,false,MCB
2025-06-04T00:00:00.000Z,274.02,285.0,274.02,282.07,1579630,false,MCB
2025-06-03T00:00:00.000Z,273.01,276.99,273.01,275.69,459027,false,MCB
2025-06-02T00:00:00.000Z,277.98,277.98,272.55,273.21,472011,false,MCB
2025-05-30T00:00:00.000Z,273.2,277.5,272.01,276.79,455631,false,MCB
2025-05-29T00:00:00.000Z,275.99,276.7,272.8,272.99,284925,false,MCB
2025-05-27T00:00:00.000Z,275.02,276.9,273.0,274.07,262638,false,MCB
2025-05-26T00:00:00.000Z,276.0,280.0,274.55,275.36,167477,false,MCB
2025-05-23T00:00:00.000Z,280.0,280.0,275.0,275.61,595250,false,MCB


In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

bronze = spark.table("psx_bronze")

w = Window.partitionBy("symbol").orderBy("date")

silver = (
    bronze
    .dropDuplicates(["symbol", "date"])
    .withColumn("daily_return_pct", 
        (F.col("close") - F.lag("close").over(w)) / F.lag("close").over(w) * 100)
    .withColumn("ma_20", F.avg("close").over(w.rowsBetween(-19, 0)))
)

silver.write.format("delta").mode("overwrite").saveAsTable("psx_silver")

In [0]:
gold = (
    spark.table("psx_silver")
    .groupBy("symbol")
    .agg(
        F.max("date").alias("latest_date"),
        F.round(F.last("close"), 2).alias("latest_close"),
        F.round(F.avg("daily_return_pct"), 3).alias("avg_daily_return_pct"),
        F.round(F.stddev("daily_return_pct"), 3).alias("volatility")
    )
)

gold.write.format("delta").mode("overwrite").saveAsTable("psx_gold")
display(gold)

symbol,latest_date,latest_close,avg_daily_return_pct,volatility
ENGRO,2025-01-03T00:00:00.000Z,485.38,0.214,1.961
LUCK,2026-08-18T00:00:00.000Z,439.3,0.054,3.887
MCB,2026-08-18T00:00:00.000Z,407.26,0.144,1.87
MEBL,2026-08-18T00:00:00.000Z,567.49,0.212,2.052
UBL,2026-08-18T00:00:00.000Z,461.98,0.197,2.999


Databricks visualization. Run in Databricks to view.